In [2]:
# import common libraires
import os
from pathlib import Path
from typing import TypedDict, Annotated, List, Optional

# import llm, parser, ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

# import langgraph libraries
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

# Rag libraries
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

load_dotenv()

c:\Users\Arbaz Khan\Desktop\Langgraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
docs = (
    PyPDFLoader("./Documents/book3.pdf").load()
)

In [4]:
chunks = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150).split_documents(docs)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

len(chunks)

1652

In [6]:
encodding = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, encodding)

In [7]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever.invoke("Gradient Descent")

[Document(id='e654c62c-b3ff-4044-afb8-fa50a6f2858d', metadata={'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2017-03-10T21:55:34+00:00', 'author': 'Aurélien Géron', 'moddate': '2017-07-29T21:43:02+08:00', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'trapped': '/False', 'source': './Documents/book3.pdf', 'total_pages': 564, 'page': 6, 'page_label': 'v'}, page_content='Batch Gradient Descent                                                                                           114\nStochastic Gradient Descent                                                                                   117\nMini-batch Gradient Descent                                                                                 119\nPolynomial Regression                                                                                                121\nLearning C

In [8]:
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0.6)
llm.invoke("What is Gradient Descent?").content

"**Gradient Descent**\n=====================\n\nGradient Descent is a popular optimization algorithm used in machine learning and deep learning to minimize the loss function of a model. It's an iterative method that adjusts the model's parameters to reduce the error between predicted and actual outputs.\n\n**How Gradient Descent Works**\n-----------------------------\n\nThe goal of Gradient Descent is to find the optimal values of the model's parameters that result in the lowest loss. The process involves the following steps:\n\n1. **Initialize Parameters**: Initialize the model's parameters (weights and biases) randomly or with pre-trained values.\n2. **Compute Loss**: Calculate the loss between the predicted output and the actual output using a loss function (e.g., Mean Squared Error or Cross-Entropy).\n3. **Compute Gradient**: Calculate the gradient of the loss function with respect to each parameter. The gradient represents the direction of the steepest ascent.\n4. **Update Paramet

### Langgraph Orchastrator

In [9]:
min_score = 0.3
max_score = 0.7

In [ ]:
class CRagState(TypedDict):
    question: str
    docs: List[Document]

    good_docs: List[Document]
    verdict: str
    reason: str
    
    strips: List[str]
    kept_strips: List[str]
    refined_context: str
    
    answer: str

In [ ]:
def retrieve_node(state: CRagState):
    q = state["question"]
    return {"docs": retriever.invoke(q)}

#### Evaluator

In [1]:
class DocEvalScore(BaseModel):
    score: float
    reason: str

doc_eval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a strict retrieval evaluator for RAG.\n"
            "You will be given ONE retrieved chunk and a question.\n"
            "Return a relevance score in [0.0, 1.0].\n"
            "- 1.0: chunk alone is sufficient to answer fully/mostly\n"
            "- 0.0: chunk is irrelevant\n"
            "Be conservative with high scores.\n"
            "Also return a short reason.\n"
            "Output JSON only.",
        ),
        ("human", "Question: {question}\n\nChunk:\n{chunk}"),
    ]
)

doc_eval_chain = doc_eval_prompt | llm.with_structured_output(DocEvalScore)

def eval_each_doc_node(state: CRagState):
    q = state["question"]

    scores: List[float] = []
    reasons: List[str] = []
    good_docs: List[Document] = []

    for d in state["docs"]:
        out = doc_eval_chain.invoke({"question": q, "chunk": d.page_content})
        scores.append(out.score)
        reasons.append(out.reason)

        if out.score > min_score:
            good_docs.append(d)
    
    if any(s > max_score for s in scores):
        return {
            "good_docs": good_docs,
            "verdict": "CORRECT",
            "reason": f"At least one retrieved chunk scored > {max_score}.",
        }
    elif len(good_docs) == 0:
        why = "No chunk was sufficient"
        return {
            "good_docs": [],
            "verdict": "INCORRECT",
            "reason": f"all retrieved chunk scored < {min_score}. {why}" 
        }
    else:
        why = "Mixed relevance signals."
        return {
            "good_docs": good_docs,
            "verdict": "AMBIGUOUS",
            "reason": f"No chunk scored > {max_score}, but not all were < {min_score}. {why}",
        }

NameError: name 'BaseModel' is not defined

#### Filtter

In [ ]:
import re
# -----------------------------
# Sentence-level DECOMPOSER
# -----------------------------
def decompose_to_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]

# -----------------------------
# FILTER (LLM judge)
# -----------------------------
class KeepOrDrop(BaseModel):
    keep: bool

filter_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a strict relevance filter.\n"
            "Return keep=true only if the sentence directly helps answer the question.\n"
            "Use ONLY the sentence. Output JSON only.",
        ),
        ("human", "Question: {question}\n\nSentence:\n{sentence}"),
    ]
)

filter_chain = filter_prompt | llm.with_structured_output(KeepOrDrop)

# -----------------------------
# REFINING (Decompose -> Filter -> Recompose)
# -----------------------------
def refine(state: CRagState):
    q = state["question"]

    # Combine retrieved docs into one context string
    # 5) In CORRECT case, eval node populates good_docs with docs having score > MIN_SCORE
    context = "\n\n".join(d.page_content for d in state["good_docs"]).strip()
    
    # 1) DECOMPOSITION: context -> sentence strips
    strips = decompose_to_sentences(context)

    # 2) FILTER: keep only relevant strips
    kept: List[str] = []
    for s in strips:
        if filter_chain.invoke({"question": q, "sentence": s}).keep:
            kept.append(s)

    # 3) RECOMPOSE: glue kept strips back together (internal knowledge)
    refined_context = "\n".join(kept).strip()

    return {
        "strips": strips,
        "kept_strips": kept,
        "refined_context": refined_context,
    }


NameError: name 'llm' is not defined

#### Generator

In [ ]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful ML tutor. Answer ONLY using the provided context.\n"
            "If the context is empty or insufficient, say: 'I don't know.'",
        ),
        ("human", "Question: {question}\n\nRefined context:\n{refined_context}"),
    ]
)

def generate(state: CRagState):
    out = (answer_prompt | llm).invoke(
        {"question": state["question"], "refined_context": state["refined_context"]}
    )
    return {"answer": out.content}

#### Incorrect and Negative Flow

In [5]:
def fail_node(state: CRagState):
    return {"answer": f"FAIL: {state['reason']}"}

def ambiguous_node(state: CRagState):
    return {"answer": f"Ambiguous: {state['reason']}"}

def route_after_eval(state: CRagState) -> str:
    if state["verdict"] == "CORRECT":
        return "refine"
    elif state["verdict"] == "INCORRECT":
        return "web_search"
    else:
        return "ambiguous"

NameError: name 'CRagState' is not defined

#### create workflow

In [ ]:
g = StateGraph(CRagState)
g.add_node("retrieve", retrieve_node)
g.add_node("eval_each_doc", eval_each_doc_node)
g.add_node("refine", refine)
g.add_node("generate", generate)
g.add_node("fail", fail_node)
g.add_node("ambiguous", ambiguous_node)


g.add_edge(START, "retrieve")
g.add_edge("retrieve", "eval_each_doc")

g.add_conditional_edges(
    "eval_each_doc",
    route_after_eval,
    {"refine": "refine", "web_search": "fail", "ambiguous": "ambiguous"}
)
g.add_edge("refine", "generate")
g.add_edge("generate", END)
g.add_edge("fail", END)

app = g.compile()
app

In [ ]:
res = app.invoke(
    {
        "question": "What are attention mechanisms and why are they important in current models?",
        "docs": [],
        "good_docs": [],
        "verdict": "",
        "reason": "",
        "strips": [],
        "kept_strips": [],
        "refined_context": "",
        "answer": "",
    }
)

print("VERDICT:", res["verdict"])
print("REASON:", res["reason"])
print("\nOUTPUT:\n", res["answer"])